In [ ]:
#Calculate T1W_sub

import os

#import visvis as vv
from PIL.ImageFilter import MedianFilter
from PIL.Image import BICUBIC
from PIL import Image,ImageOps
from skimage.io import imread, imshow, imsave
from skimage import io
import skimage
import torchvision.transforms as transforms
from pydicom import dcmread
import segmentation_models_pytorch as smp
from collections import namedtuple
import nibabel as nib
import imageio
from torchvision.transforms import CenterCrop
from scipy import stats


#from monai.losses.ssim_loss import SSIMLoss #Valid for 3D
#from monai.losses import MultiScaleLoss
import numpy as np
#import visvis as vv



In [ ]:

def eliminate_outliers(im):
    im[im<0]=0
    im[im>stats.percentileofscore(im,99.9)]=stats.percentileofscore(im,99.9)
    return im



def change_DR(modify, template):
    DR=np.mean(template)/np.max(template)
    print(DR)
    im_after_DR=DR*modify
    return im_after_DR
def find_DR(template):
    dr_lambda=np.mean(template)/np.max(template)
    print(dr_lambda)
    return dr_lambda

def show_image(im):
    pilIm=PIL.Image.create(im.astype('int'))
    pilIm.show(cmap='gray',vmin=0, vmax=255)



In [ ]:
#Calculate T1 subtraction
path_input=r'DATA_ROOT\3_StructuralPreProcessing\preprocesado_results_v2'
path_output=r'DATA_ROOT\3_StructuralPreProcessing\volume_slices_all\T1W_SUB_PYTHON\Train'
path_output_2=r'DATA_ROOT\3_StructuralPreProcessing\volume_slices_all\T1W_SUB_PYTHON_unnormalized\Train'


t1_ce_path=os.path.join(path_input,'T1W_CE_3D_FFE_TRA_IQ')
t1_path=os.path.join(path_input,'T1W_3D_FFE_TRA_IQ')

it=0
for pat in range(51,87):
    it=it+1
    pat_path=os.path.join(path_output, 'PAT_'+str(pat)+'_')
    pat_path_2=os.path.join(path_output_2, 'PAT_'+str(pat)+'_')

    if not(os.path.isdir(pat_path)):
        os.makedirs(pat_path)
    if not(os.path.isdir(pat_path_2)):
        os.makedirs(pat_path_2)
    t1_ce_im=os.path.join(t1_ce_path,'PAT_'+str(pat)+'_','PAT_'+str(pat)+'_T1W_CE_3D_FFE_TRA_IQ_final_crop.nii.gz')
    t1_im=os.path.join(t1_path,'PAT_'+str(pat)+'_','PAT_'+str(pat)+'_T1W_3D_FFE_TRA_IQ_final_crop.nii.gz')
    t1_nib=nib.load(t1_im)
    t1_info=t1_nib.get_fdata()
    t1_read=eliminate_outliers(t1_info)
    
    t1_ce_nib=nib.load(t1_ce_im)
    t1_ce_info=t1_ce_nib.get_fdata()
    t1_ce_read=eliminate_outliers(t1_ce_info)
    
    #t1_after_DR=change_DR(t1_read,t1_ce_read)
    t1_after_DR=find_DR(t1_ce_read)*t1_read
    print(t1_read.max())
    print(t1_after_DR.max())
    #t1_before_DR_im=show_image(t1_read[:,:,90])
    #t1_after_DR_im=show_image( t1_after_DR[:,:,90])
   
    sub_im=np.abs(t1_read/np.mean(t1_read)-t1_ce_read/np.mean(t1_ce_read))
    sub_im_normal=(sub_im-np.min(sub_im))/(np.max(sub_im)-np.min(sub_im))
   
    slice_eg=sub_im_normal[:,:,130]
    n_slices=sub_im_normal.shape[2]
    for j in range(1,n_slices):
        new_path=os.path.join(pat_path, 'PAT_'+str(pat)+'_T1W_SUB_PYTHON_slice_'+str(j)+'.nii.gz')
        path_2=os.path.join(pat_path_2, 'PAT_'+str(pat)+'_T1W_SUB_PYTHON_unnormalized_slice_'+str(j)+'.nii.gz')
        
        ni_img=nib.Nifti1Image(sub_im_normal[:,:,j], t1_ce_nib.affine)
        ni_img_2=nib.Nifti1Image(sub_im[:,:,j], t1_ce_nib.affine)
        nib.save(ni_img, new_path)
        #nib.save(ni_img_2,path_2)
        
        #im=show_image(t1_after_DR[:,:,j])
        #vasos=show_image(sub_im[:,:,j])

  
    
        

In [ ]:
path_1=r'DATA_ROOT\3-StructuralPreProcessing\resultados_preprocesado_prueba\volume_slices\T1W_CE_3D_FFE_TRA_IQ\Train\PAT_1_\PAT_1_T1W_CE_3D_FFE_TRA_IQ_slice_90.nii.gz'
path_2=r'DATA_ROOT\3-StructuralPreProcessing\resultados_preprocesado_prueba\volume_slices\T1W_3D_FFE_TRA_IQ\Train\PAT_1_\PAT_1_T1W_3D_FFE_TRA_IQ_slice_90.nii.gz'

im_nib=nib.load(path_1)
im_info=im_nib.get_fdata()

im2_nib=nib.load(path_2)
im2_info=im2_nib.get_fdata()

no_outlier_t1ce=eliminate_outliers(im_info)
no_outlier_t1=eliminate_outliers(im2_info)

#new_im=change_DR(no_outlier_t1,no_outlier_t1ce)
new_im=change_DR(im2_info,im_info)


print('T1 image before applying T1_CE DR')
print(im2_info.max())
print(im2_info.min())
pilIm_before=PILImage.create(im2_info.astype('int'))
pilIm_before.show(cmap='gray')


print('T1 image after applying T1_CE DR')
print(new_im.max())
print(new_im.min())
pilIm=show_image(new_im)



In [ ]:
#Calculate T1 subtraction without absolute value
path_input=r'DATA_ROOT\3_StructuralPreProcessing\resultados_preprocesado_prueba'
path_output=r'DATA_ROOT\3_StructuralPreProcessing\volume_slices\volume_slices\T1W_SUB_PYTHON_2\Valid'
path_output_2=r'DATA_ROOT\3_StructuralPreProcessing\volume_slices\volume_slices\T1W_SUB_PYTHON_unnormalized_2\Valid'


t1_ce_path=os.path.join(path_input,'T1W_CE_3D_FFE_TRA_IQ')
t1_path=os.path.join(path_input,'T1W_3D_FFE_TRA_IQ')

it=0
for pat in range(33,41):
    it=it+1
    pat_path=os.path.join(path_output, 'PAT_'+str(pat)+'_')
    pat_path_2=os.path.join(path_output_2, 'PAT_'+str(pat)+'_')

    if not(os.path.isdir(pat_path)):
        os.makedirs(pat_path)
    if not(os.path.isdir(pat_path_2)):
        os.makedirs(pat_path_2)
    t1_ce_im=os.path.join(t1_ce_path,'PAT_'+str(pat)+'_','PAT_'+str(pat)+'_T1W_CE_3D_FFE_TRA_IQ_final_crop.nii.gz')
    t1_im=os.path.join(t1_path,'PAT_'+str(pat)+'_','PAT_'+str(pat)+'_T1W_3D_FFE_TRA_IQ_final_crop.nii.gz')
    t1_nib=nib.load(t1_im)
    t1_info=t1_nib.get_fdata()
    t1_read=eliminate_outliers(t1_info)
    
    t1_ce_nib=nib.load(t1_ce_im)
    t1_ce_info=t1_ce_nib.get_fdata()
    t1_ce_read=eliminate_outliers(t1_ce_info)
    
    #t1_after_DR=change_DR(t1_read,t1_ce_read)
    t1_after_DR=find_DR(t1_ce_read)*t1_read
    print(t1_read.max())
    print(t1_after_DR.max())
    t1_before_DR_im=show_image(t1_read[:,:,90])
    t1_after_DR_im=show_image( t1_after_DR[:,:,90])
   
    sub_im=t1_read/np.mean(t1_read)-t1_ce_read/np.mean(t1_ce_read)
    sub_im_normal=(sub_im-np.min(sub_im))/(np.max(sub_im)-np.min(sub_im))
   
    slice_eg=sub_im_normal[:,:,130]
    n_slices=sub_im_normal.shape[2]
    for j in range(1,n_slices):
        new_path=os.path.join(pat_path, 'PAT_'+str(pat)+'_T1W_SUB_PYTHON_2_slice_'+str(j)+'.nii.gz')
        path_2=os.path.join(pat_path_2, 'PAT_'+str(pat)+'_T1W_SUB_PYTHON_unnormalized_2_slice_'+str(j)+'.nii.gz')
        
        ni_img=nib.Nifti1Image(sub_im_normal[:,:,j], t1_ce_nib.affine)
        ni_img_2=nib.Nifti1Image(sub_im[:,:,j], t1_ce_nib.affine)
        #nib.save(ni_img, new_path)
        #nib.save(ni_img_2,path_2)
        
        #im=show_image(t1_after_DR[:,:,j])
        #vasos=show_image(sub_im[:,:,j])


In [ ]:
#Paint T1-Method Laura
path_input=r'DATA_ROOT\3-StructuralPreProcessing\resultados_preprocesado_prueba\volume_slices'
path_output=r'DATA_ROOT\3-StructuralPreProcessing\resultados_preprocesado_prueba\volume_slices\T1W_SUB_PYTHON\Valid' #en este caso es el path_vasos_input
path_vasos_output=r'DATA_ROOT\4-DL\results\MRISyth_Local_res50-17072023_bestEpoch'


t1_ce_path=os.path.join(path_input,'T1W_CE_3D_FFE_TRA_IQ', 'Valid')
t1_path=os.path.join(path_input,'T1W_3D_FFE_TRA_IQ', 'Valid')


for pat in range(33,34):
    pat_path=os.path.join(path_output, 'PAT_'+str(pat)+'_')
    vasos_path_output=os.path.join(path_vasos, 'PAT_'+str(pat)+'_.nii.gz')
    if not(os.path.isdir(pat_path)):
        os.makedirs(pat_path)
    
    
    n_slices=len([name for name in os.listdir(pat_path)])
    print(n_slices)
    for j in range(1,n_slices):
        print(j)
        t1_ce_im=os.path.join(t1_ce_path,'PAT_'+str(pat)+'_','PAT_'+str(pat)+'_T1W_CE_3D_FFE_TRA_IQ_slice_'+str(j)+'.nii.gz')
        t1_im=os.path.join(t1_path,'PAT_'+str(pat)+'_','PAT_'+str(pat)+'_T1W_3D_FFE_TRA_IQ_slice_'+str(j)+'.nii.gz')
        t1_nib=nib.load(t1_im)
        t1_read=t1_nib.get_fdata()
        
        #t1_clean=eliminate_outliers(t1_read)
        #t1_adjusted=t1_clean/np.mean(t1_clean)
        #print(t1_adjusted.mean())
        #Normalize t1
        t1_read_normal=(t1_read-np.min(t1_read))/(np.max(t1_read)-np.min(t1_read))
        #t1_read_normal=(t1_clean-np.min(t1_clean))/(np.max(t1_clean)-np.min(t1_clean))
        #print(t1_read_normal.shape)

        t1_ce_nib=nib.load(t1_ce_im)
        t1_ce_read=t1_ce_nib.get_fdata()
        
        #t1_ce_clean=eliminate_outliers(t1_ce_read)
        #t1_ce_adjusted=t1_ce_clean/np.mean(t1_ce_clean)
        
        #print(t1_ce_adjusted.mean())
        #Normalize t1_ce and find mean value
        t1_ce_read_normal=(t1_ce_read-np.min(t1_ce_read))/(np.max(t1_ce_read)-np.min(t1_ce_read))
        #t1_ce_read_normal=(t1_ce_clean-np.min(t1_ce_clean))/(np.max(t1_ce_clean)-np.min(t1_ce_clean))

        dr_t1_ce=np.mean(t1_ce_read_normal)/np.mean(t1_read_normal)
        print('Mean t1 normal:')
        print(t1_read_normal.mean())
        print('Mean t1_ce normal:')
        print(t1_ce_read_normal.mean())
        print('DR t1_ce:')
        print(dr_t1_ce)
        t1_after_DR=dr_t1_ce*t1_read_normal
        #t1_normal=show_image(t1_read_normal)
        #t1_show=show_image(t1_read)
        print('Mean t1 adjusted:')
        print(t1_after_DR.mean())
        
        sub_im=np.abs(t1_read/np.mean(t1_read)-t1_ce_read/np.mean(t1_ce_read))
        sub_im_normal=(sub_im-np.min(sub_im))/(np.max(sub_im)-np.min(sub_im))
        #vasos_show=show_image(sub_input)
        
   
        sub_output_nib=nib.load(vasos_path_output)
        sub_output=sub_output_nib.get_fdata()
        sub_output_normal=(sub_output-np.min(sub_output))/(np.max(sub_output)-np.min(sub_output))
        dr_vasos=np.mean(sub_im_normal)/np.mean(sub_output_normal[:,:,j])
        print('DR vasos:')
        print(dr_vasos)
        vasos_after_DR=dr_vasos*sub_output_normal[:,:,j]
        print('Mean vasos adjusted')
        print(vasos_after_DR.mean())
        
        
        final_img=np.add(t1_after_DR,vasos_after_DR)
        print('Mean final img:')
        print(final_img.mean())
        show_img=show_image(final_img)
    
    
    
        

In [ ]:
#Paint T1-Method Dani
path_input=r'DATA_ROOT\3-StructuralPreProcessing\resultados_preprocesado_prueba\volume_slices'
path_output=r'DATA_ROOT\3-StructuralPreProcessing\resultados_preprocesado_prueba\volume_slices\T1W_SUB_PYTHON\Valid' #en este caso es el path_vasos_input
path_vasos_output=r'DATA_ROOT\4-DL\results\MRISyth_Local_res50-17072023_bestEpoch'


t1_ce_path=os.path.join(path_input,'T1W_CE_3D_FFE_TRA_IQ', 'Valid')
t1_path=os.path.join(path_input,'T1W_3D_FFE_TRA_IQ', 'Valid')


for pat in range(33,34):
    pat_path=os.path.join(path_output, 'PAT_'+str(pat)+'_')
    vasos_path_output=os.path.join(path_vasos, 'PAT_'+str(pat)+'_.nii.gz')
    if not(os.path.isdir(pat_path)):
        os.makedirs(pat_path)
    
    
    n_slices=len([name for name in os.listdir(pat_path)])
    print(n_slices)
    for j in range(1,n_slices):
        t1_ce_im=os.path.join(t1_ce_path,'PAT_'+str(pat)+'_','PAT_'+str(pat)+'_T1W_CE_3D_FFE_TRA_IQ_slice_'+str(j)+'.nii.gz')
        t1_im=os.path.join(t1_path,'PAT_'+str(pat)+'_','PAT_'+str(pat)+'_T1W_3D_FFE_TRA_IQ_slice_'+str(j)+'.nii.gz')
        t1_nib=nib.load(t1_im)
        t1_read=t1_nib.get_fdata()
        
        #t1_clean=eliminate_outliers(t1_read)
        t1_adjusted=t1_read/np.mean(t1_read)
        #print(t1_adjusted.mean())
        #Normalize t1
        #t1_read_normal=(t1_read-np.min(t1_read))/(np.max(t1_read)-np.min(t1_read))
        #print(t1_read_normal.shape)

        t1_ce_nib=nib.load(t1_ce_im)
        t1_ce_read=t1_ce_nib.get_fdata()
        
        #t1_ce_clean=eliminate_outliers(t1_ce_read)
        t1_ce_adjusted=t1_ce_read/np.mean(t1_ce_read)
        
        #print(t1_ce_adjusted.mean())
        #Normalize t1_ce and find mean value
        #t1_ce_read_normal=(t1_ce_read-np.min(t1_ce_read))/(np.max(t1_ce_read)-np.min(t1_ce_read))
        print('Mean t1 adjsuted:')
        print(t1_adjusted.mean())
        print('Mean t1_ce adjusted:')
        print(t1_ce_adjusted.mean())
        print('Mean t1:')
        print(t1_read.mean())
        print('Mean t1_ce:')
        print(t1_ce_read.mean())
        dr_t1_ce=t1_ce_read.mean()/t1_read.mean()
        #print(dr_t1_ce)
        t1_after_DR=dr_t1_ce*t1_adjusted
        t1_show=show_image(t1_after_DR)
        
        
        sub_im=np.abs(t1_clean/np.mean(t1_clean)-t1_ce_clean/np.mean(t1_ce_clean))
        vasos_show=show_image(sub_input)
        
   
        sub_output_nib=nib.load(vasos_path_output)
        sub_output=sub_output_nib.get_fdata()
        sub_output_normal=(sub_output-np.min(sub_output))/(np.max(sub_output)-np.min(sub_output))
        
        final_img=np.add(t1_after_DR,sub_output_normal[:,:,j]*sub_input.max())
        #show_img=show_image(final_img)
    
    
    
        

In [ ]:
#Paint T1-Method Dani 2
path_input=r'DATA_ROOT\3-StructuralPreProcessing\resultados_preprocesado_prueba\volume_slices'
path_output=r'DATA_ROOT\3-StructuralPreProcessing\resultados_preprocesado_prueba\volume_slices\T1W_SUB_PYTHON\Valid' #en este caso es el path_vasos_input
path_vasos_output=r'DATA_ROOT\4-DL\results\MRISyth_Local_res50-17072023_bestEpoch'
path_vasos_unnormalized=r'DATA_ROOT\3-StructuralPreProcessing\resultados_preprocesado_prueba\volume_slices\T1W_SUB_PYTHON_unnormalized\Valid'


t1_ce_path=os.path.join(path_input,'T1W_CE_3D_FFE_TRA_IQ', 'Valid')
t1_path=os.path.join(path_input,'T1W_3D_FFE_TRA_IQ', 'Valid')


for pat in range(33,34):
    pat_path=os.path.join(path_output, 'PAT_'+str(pat)+'_')
    vasos_path_output=os.path.join(path_vasos_output, 'PAT_'+str(pat)+'_.nii.gz')
    if not(os.path.isdir(pat_path)):
        os.makedirs(pat_path)
    
    
    n_slices=len([name for name in os.listdir(pat_path)])
    print(n_slices)
    for j in range(1,n_slices):
        t1_ce_im=os.path.join(t1_ce_path,'PAT_'+str(pat)+'_','PAT_'+str(pat)+'_T1W_CE_3D_FFE_TRA_IQ_slice_'+str(j)+'.nii.gz')
        t1_im=os.path.join(t1_path,'PAT_'+str(pat)+'_','PAT_'+str(pat)+'_T1W_3D_FFE_TRA_IQ_slice_'+str(j)+'.nii.gz')
        t1_nib=nib.load(t1_im)
        t1_read=t1_nib.get_fdata()
        print('Mean t1:')
        print(np.mean(t1_read))
        
        t1_clean=eliminate_outliers(t1_read)
        print('Mean t1 w/o outliers:')
        print(np.mean(t1_clean))
        
        
        #print(t1_adjusted.mean())
        #Normalize t1
        #t1_read_normal=(t1_read-np.min(t1_read))/(np.max(t1_read)-np.min(t1_read))
        #print(t1_read_normal.shape)

        t1_ce_nib=nib.load(t1_ce_im)
        t1_ce_read=t1_ce_nib.get_fdata()
        print('Mean t1_ce')
        print(np.mean(t1_ce_read))
        
        t1_ce_clean=eliminate_outliers(t1_ce_read)
        print('Mean t1_ce without outliers')
        print(np.mean(t1_ce_clean))
              
        dr_t1_ce=t1_ce_read.mean()/t1_read.mean()
        print('DR t1_ce')
        print(dr_t1_ce)
              
        t1_adjusted=(t1_read*dr_t1_ce).astype('uint32')
        print('Mean t1 adjsuted:')
        print(t1_adjusted.mean())

    
        
        path_vasos_input=os.path.join(path_vasos_unnormalized,'PAT_'+str(pat)+'_','PAT_'+str(pat)+'_T1W_SUB_PYTHON_unnormalized_slice_'+str(j)+'.nii.gz')
        sub_im_nib=nib.load(path_vasos_input)
        sub_im=sub_im_nib.get_fdata()
        #vasos_show=show_image(sub_im)
                            
        sub_output_nib=nib.load(vasos_path_output)
        sub_output=sub_output_nib.get_fdata()
        sub_output_normal=(sub_output-np.min(sub_output))/(np.max(sub_output)-np.min(sub_output))
        #factor=np.mean(t1_adjusted)/np.mean(sub_output[:,:,j]*sub_im.max())
        

        print('Mean vasos ajustados:')
        print((sub_output[:,:,j]*sub_im.max()).mean())
        print('max vasos_input')
        print(sub_im.max())

        final_img=np.add((t1_adjusted).astype('uint32'),(sub_output_normal[:,:,j]*sub_im.max()).astype('uint32'))
        print('Mean final image:')
        print(final_img.mean())
        show_img=show_image(final_img.astype('uint32'))
    
    
    
        

In [ ]:
#Paint T1-Method Laura 2
path_input=r'DATA_ROOT\3-StructuralPreProcessing\resultados_preprocesado_prueba\volume_slices'
path_output=r'DATA_ROOT\3-StructuralPreProcessing\resultados_preprocesado_prueba\volume_slices\T1W_SUB_PYTHON\Valid' #en este caso es el path_vasos_input
path_vasos_output=r'DATA_ROOT\4-DL\results\MRISyth_Local_res50-17072023_bestEpoch'


t1_ce_path=os.path.join(path_input,'T1W_CE_3D_FFE_TRA_IQ', 'Valid')
t1_path=os.path.join(path_input,'T1W_3D_FFE_TRA_IQ', 'Valid')


for pat in range(33,34):
    pat_path=os.path.join(path_output, 'PAT_'+str(pat)+'_')
    vasos_path_output=os.path.join(path_vasos, 'PAT_'+str(pat)+'_.nii.gz')
    if not(os.path.isdir(pat_path)):
        os.makedirs(pat_path)
    
    
    n_slices=len([name for name in os.listdir(pat_path)])
    for j in range(1,n_slices):
        t1_ce_im=os.path.join(t1_ce_path,'PAT_'+str(pat)+'_','PAT_'+str(pat)+'_T1W_CE_3D_FFE_TRA_IQ_slice_'+str(j)+'.nii.gz')
        t1_im=os.path.join(t1_path,'PAT_'+str(pat)+'_','PAT_'+str(pat)+'_T1W_3D_FFE_TRA_IQ_slice_'+str(j)+'.nii.gz')
        t1_nib=nib.load(t1_im)
        t1_read=t1_nib.get_fdata()
        
        #Normalize t1
        t1_read_normal=(t1_read-np.min(t1_read))/(np.max(t1_read)-np.min(t1_read))
        #print(t1_read_normal.shape)

        t1_ce_nib=nib.load(t1_ce_im)
        t1_ce_read=t1_ce_nib.get_fdata()
        
        
        print(t1_ce_adjusted.mean())
         #Normalize t1_ce and find mean value
        t1_ce_read_normal=(t1_ce_read-np.min(t1_ce_read))/(np.max(t1_ce_read)-np.min(t1_ce_read))
        dr_t1_ce=t1_ce_read_normal.mean()/t1_read_normal.mean()
        print(dr_t1_ce)
        t1_after_DR=dr_t1_ce*t1_read_normal
        print('MAX t1_dr:')
        print(t1_after_DR.max())
        
        
        sub_im=np.abs(t1_read/np.mean(t1_read)-t1_ce_read/np.mean(t1_ce_read))
        #print('Max vasos:')
        #print(sub_input.max())
        sub_im_normal=(sub_im-np.min(sub_im))/(np.max(sub_im)-np.min(sub_im))
       
        #print(dr_t1_ce)

        #t1_before_DR_im=show_image(t1_read_normal)
        #t1_ce_img=show_image(t1_ce_read_normal)
        t1_after_DR_im=show_image( t1_after_DR)
    
        
        #vasos_path_input=os.path.join(pat_path, 'PAT_'+str(pat)+'_T1W_SUB_PYTHON_slice_'+str(j)+'.nii.gz')
        #sub_input_nib=nib.load(vasos_path_input)
        #sub_input_info=sub_input_nib.get_fdata()
        
   
        sub_output_nib=nib.load(vasos_path_output)
        sub_output=sub_output_nib.get_fdata()
        sub_output_normal=(sub_output[:,:,j]-np.min(sub_output[:,:,j]))/(np.max(sub_output[:,:,j])-np.min(sub_output[:,:,j]))
        #sub_output=eliminate_outliers(sub_output)
        #sub_output_normal=(sub_output-np.min(sub_output))/(np.max(sub_output)-np.min(sub_output))
        dr_vasos=sub_im_normal.mean()/sub_output_normal.mean()
        vasos_after=dr_vasos*sub_output_normal
        #print(vasos_after.shape)
        #im_vasos=show_image(vasos_after)

        #show_img=show_image(sub_output[:,:,j])
        #final_img=t1_after_DR+sub_output[:,:,j]*sub_input.max()
        final_img=t1_after_DR+sub_output_normal
        #final_img_clean=eliminate_outliers(final_img)
        show_img=show_image(final_img)
    
    
    
        

In [ ]:
#Paint T1
path_input=r'DATA_ROOT\3-StructuralPreProcessing\resultados_preprocesado_prueba\volume_slices'
path_output=r'DATA_ROOT\3-StructuralPreProcessing\resultados_preprocesado_prueba\volume_slices\T1W_SUB_PYTHON\Valid' #en este caso es el path_vasos_input
path_vasos_output=r'DATA_ROOT\4-DL\results\MRISyth_Local_res50-17072023_bestEpoch'


t1_ce_path=os.path.join(path_input,'T1W_CE_3D_FFE_TRA_IQ', 'Valid')
t1_path=os.path.join(path_input,'T1W_3D_FFE_TRA_IQ', 'Valid')


for pat in range(33,34):
    pat_path=os.path.join(path_output, 'PAT_'+str(pat)+'_')
    vasos_path_output=os.path.join(path_vasos, 'PAT_'+str(pat)+'_.nii.gz')
    if not(os.path.isdir(pat_path)):
        os.makedirs(pat_path)
    
    
    n_slices=len([name for name in os.listdir(pat_path)])
    print(n_slices)
    for j in range(1,n_slices):
        t1_ce_im=os.path.join(t1_ce_path,'PAT_'+str(pat)+'_','PAT_'+str(pat)+'_T1W_CE_3D_FFE_TRA_IQ_slice_'+str(j)+'.nii.gz')
        t1_im=os.path.join(t1_path,'PAT_'+str(pat)+'_','PAT_'+str(pat)+'_T1W_3D_FFE_TRA_IQ_slice_'+str(j)+'.nii.gz')
        t1_nib=nib.load(t1_im)
        t1_info=t1_nib.get_fdata()
        #t1_read=eliminate_outliers(t1_info)
        #Normalize t1
        t1_read_normal=(t1_read-np.min(t1_read))/(np.max(t1_read)-np.min(t1_read))
        #print(t1_read_normal.shape)

        t1_ce_nib=nib.load(t1_ce_im)
        t1_ce_info=t1_ce_nib.get_fdata()
        #t1_ce_read=eliminate_outliers(t1_ce_info)
         #Normalize t1_ce and find mean value
        t1_ce_read_normal=(t1_ce_read-np.min(t1_ce_read))/(np.max(t1_ce_read)-np.min(t1_ce_read))
        dr_t1_ce=t1_ce_read_normal.mean()/t1_read_normal.mean()

        t1_after_DR=dr_t1_ce*t1_read_normal

        #print(dr_t1_ce)

        #t1_before_DR_im=show_image(t1_read_normal)
        #t1_ce_img=show_image(t1_ce_read_normal)
        #t1_after_DR_im=show_image( t1_after_DR)
    
        sub_input=
        vasos_path_input=os.path.join(pat_path, 'PAT_'+str(pat)+'_T1W_SUB_PYTHON_slice_'+str(j)+'.nii.gz')
        sub_input_nib=nib.load(vasos_path_input)
        sub_input_info=sub_input_nib.get_fdata()
        
   
        sub_output_nib=nib.load(vasos_path_output)
        sub_output=sub_output_nib.get_fdata()
        sub_output=eliminate_outliers(sub_output)
        sub_output_normal=(sub_output-np.min(sub_output))/(np.max(sub_output)-np.min(sub_output))
        dr_vasos=sub_output_normal.mean()/sub_input_info.mean()
        vasos_after=dr_vasos*sub_output_normal[:,:,j]
        #print(vasos_after.shape)
        im_vasos=show_image(vasos_after)
        final_img=t1_after_DR+vasos_after
        show_img=show_image(final_img)
    
        